# 第14回：機械学習入門（講義）
担当：相澤宏旭

## 内容
第12回目の講義では，Pythonを用いてデータを読み込み，可視化し，そしてデータの性質を説明する統計量を算出した．平均や分散といった統計量もデータの性質を我々が分析するうえで重要な手がかりとなるが，このような手がかりをコンピュータ自らがデータから学ぶことができないだろうか？そして，データと手がかりから対象の性質を予測することはできないだろうか？このような動機のもと，**機械学習（Machine Learning）** と呼ばれる研究分野は **人工知能（Artificial Intelligence）** の一分野として発展してきた．

特に，近年，機械学習技術の進展は著しい．突然だが，ゼロからはじめるプログラミングのサポートページにある猫の画像（下の図）を見てほしい．実は，この画像も機械学習技術を用いて生成したものである．作成時には，文章から画像を生成するモデル [DALL·E 3](https://openai.com/dall-e-3) に，**Minimalistic illustration of a cat programming on a laptop, suitable for a web page icon.** という文章を与えた．これは，「ノートパソコンでプログラミングする猫の，Webページのアイコンに適したシンプルなイラスト」という意味である．

このような画像生成では，大量のデータから学習した画像と言葉の関係を利用して，入力された文章に応じた画像を作り出す．機械学習技術は，数値の予測だけでなく，このような画像の生成にも利用されている．

![](./figs/gen_cats.png)

```{admonition} 機械学習の研究や実装に触れるには
:class: tip

新しい機械学習技術の研究成果は[arXiv](https://arxiv.org/)と呼ばれる，論文のプレプリント（査読前の原稿など）を公開するリポジトリにも掲載されている．論文を読み解いて実装するのは大変であるが，研究によってはPythonで書かれたコードがGitHubで公開されていたり，[Hugging Face](https://huggingface.co/)と呼ばれるサービスでモデルやデモを試せたりする．
```

このように非常にエキサイティングな研究分野であるが，その仕組みは複雑である．本講義では説明を簡略化し，実際にプログラムを動かしながら機械学習の雰囲気を掴むことを目標とする．具体的には，以下を学ぶ．

- 機械学習の概要を学ぶ．
- 線形回帰をscikit-learnで実装する．
- 【発展】NumPyのみで線形回帰を実装する．

## 機械学習

まずは機械学習の雰囲気を掴むために，基本的な考え方と用語を学ぼう．

本講義では，**タスク**（例：画像に写る対象を判別する，気温から売り上げを予測する）を行うための **モデル** を，パラメータ $\theta$ を持つ関数 $f_{\theta}$ として考える．パラメータとは，関数の振る舞いを決める値である．後で扱う直線の式では，傾きや切片がこれにあたる．

ここでモデルを関数として考えるとは，下図に示すように，何らかの入力 $x$ から，タスクに対応する出力 $y$ を計算する関数 $y=f_{\theta}(x)$ を考えることである．関数の形を決め，データを使ってパラメータを求めることが，ここで扱うモデルの学習となる．詳細は後述する．

![](./figs/overview_ml.png)

具体例を考えるとわかりやすい．上図では，ハスキーの画像を入力し，それぞれの候補に対する予測確率を出力する関数を考える．最も高い確率の候補を予測結果とすると，画像とそこに写る対象の種類の関係を関数 $f_{\theta}$ でモデル化した **物体認識** の例となる．

また，第12回の例のように入力が気温，出力がアイスクリームの売り上げであれば，関数 $y=f_{\theta}(x)$ は，気温と売り上げの関係をモデル化したものとなる．学習したモデルに新しい入力 $x'$ を与えると，対応する出力を **予測** できる．例えば，まだ見ていない画像に写る対象や，新たな気温に対する売り上げを予測する場面で利用できる．

このように，機械学習に解かせたい問題を **タスク** と呼ぶ．後述する教師あり学習の代表的なタスクには，**分類** と **回帰** がある．

分類は，入力 $x$ がどの種類に属するかを予測するタスクである．分類先の種類を **クラス**，その種類を表す名前や番号を **ラベル** と呼ぶ．上図のように，画像に写る犬の種類を判別することが具体例となる．一方で，回帰は，入力 $x$ に対応する売り上げや体重などの数値を予測するタスクである．$x$ と $y$ を身長と体重，数学とプログラミングの成績，第12回のような気温とアイスクリームの売り上げで考えたときの，$x$ から $y$ の予測は回帰タスクの例である．

```{admonition} 分類でいう「クラス」
:class: tip

ここでのクラスは，分類先の種類を意味する．第10回で学んだ，`class` を使ってデータと処理をまとめるPythonのクラスとは，別の意味で使われている．
```

そして，機械学習技術のおもしろさは，タスクを解くためのモデルをデータから統計的に求めることにある．ここでは，代表的な学習の方法として **教師あり学習** と **教師なし学習** を紹介する．両者の違いは，入力に対応する正解の情報を使うかどうかにあり，どのような情報からモデルを学習するかがポイントとなる．

- 教師あり学習では，入力と，それに対応する正解の情報を組にした **教師データ** を利用する．モデルの予測と正解の違いが小さくなるように，パラメータを求める．
    ![](./figs/supervised.png)
- 教師なし学習では，入力に対応する正解の情報を与えずに，データの構造や特性を捉える．例えば，似た特徴を持つデータをグループに分ける方法がある．
    ![](./figs/unsupervised.png)

このように，何をしたいのかというタスクと，どのような情報を学習に使えるのかを整理することが，機械学習において重要となる．

本講義では，機械学習の基礎として **線形回帰（Linear Regression）** という技術をPythonを使って実装することを試みる．
線形回帰は教師あり学習の一種であり，名前の通り，回帰タスクを解くアルゴリズムである．

また，機械学習モデルにはさまざまな形がある．線形回帰と **ニューラルネットワーク** では，関数の構造や学習方法などに違いがあるが，入力と出力の関係をモデルで表し，データからパラメータを学ぶという考え方は共通している．

```{admonition} 本講義で扱う範囲
:class: caution

上記の機械学習の説明は非常に簡略化された説明であり，具体例も限定的である点に留意されたい．機械学習については専門科目または[さらに勉強したい人へ](#advanced-ml)に参考資料を記載したので機械学習に興味がある学生は参照されたい．
```

## 線形回帰
前述したように，**回帰（Regression）** とは，入力 $x$ に対応する売り上げや体重などの数値を予測するタスクである．線形回帰では，入力 $x$ と，それに対応する正解データ $y$ を利用する教師あり学習を行う．まずは，第2回と同様に $y=2x+1$ の式を例にしながら，回帰タスクを解くための関数 $f_{\theta}$ を説明する．

このときのゴールは，データから入力と出力の関係を求め，新しい入力 $x'$ に対する値を予測することである．関数の形が $y=ax+b$ で，傾き $a=2$ と切片 $b=1$ が既にわかっていれば，計算は容易である．例えば，$y=2x+1$ の場合，$x'=10$ ならば対応する値は $21$ となる．

しかしながら，もし傾き $a$ と切片 $b$ がわからなかったらどうだろうか？左図のように，データの傾向を表す関数の候補はいくつも考えられる．どのような基準で，データによく当てはまる関数を決めれば良いだろうか？

![](./figs/linear_regression.png)

ここでは，線形回帰のモデルとして，以下のような直線を表す関数，

$$
f_{\theta}(x) = ax + b,
$$

を仮定する．関数の形はこのように決めておき，傾きと切片をパラメータ $\theta=\{a,b\}$ として，データから求める．そのために，入力 $x$ と対応する出力 $y$ のペア $(x,y)$ を複数用意する．

本講義で扱う線形回帰では，直線による予測値 $ax+b$ と各データの観測値 $y$ の差を二乗し，その合計が最小になるように $a,b$ を求める．この方法を **最小二乗法** と呼ぶ．差をそのまま足すと正と負が打ち消し合うため，二乗してから合計する．計算方法の詳細は，後半の発展部分で紹介する．

一度パラメータを求めれば，新しい入力 $x'$ に対しても，同じ式 $ax'+b$ を使って値を予測できる．求めたパラメータ $a,b$ は，入力 $x$ と出力 $y$ の関係を表しており，データの背後にある関係性をペア $(x,y)$ から推定したとも言える．

```{admonition} 予測値と実際の値は必ずしも一致しない
:class: tip

実際のデータにはばらつきがあるため，一本の直線がすべての観測値を通るとは限らない．線形回帰では，データ全体によく当てはまる直線を求める．また，新しい入力に対する予測値が，実際に観測される値と必ず一致するわけではない．
```

$y=2x+1$ という例だけでは利用場面をイメージすることが難しいだろうが，$x$ と $y$ をそれぞれ，身長と体重，数学とプログラミングの成績，第12回のような気温とアイスクリームの売り上げなどで考えると実用例が見えてくる．当日の気温 $x$ から売り上げ $y$ が予測できれば，在庫管理にも役立つだろう．

最初の画像生成の例とは見た目のインパクトに違いがあるが，データのパターンをパラメータとして表現し，データから学習するという考え方は共通している．線形回帰は機械学習の基礎として多くの教科書で紹介される手法であるため，Pythonによる機械学習プログラミングの第一歩として体験してほしい．

以降，傾き $a$ と切片 $b$ をPythonを使って求める．


## scikit-learnによる線形回帰
本講義では，計算と機械学習のために，以下の二つのライブラリを用いる．グラフの表示には，第12回でも利用したMatplotlibを使う．

- [NumPy](https://numpy.org)は，複数の値を持つデータの計算をサポートするライブラリである．ベクトルや行列の計算が容易で，逆行列や固有値の計算など，線形代数で学ぶ演算も用意されている．公式ドキュメントの[Learn](https://numpy.org/learn/)に，基礎的な使い方を学べる資料がまとまっている．

- [scikit-learn](https://scikit-learn.org/stable/)は，本講義で扱う回帰だけでなく，分類，クラスタリング，次元削減など，様々な機械学習やデータ分析のアルゴリズムが実装されているライブラリである．公式ドキュメントの[Examples](https://scikit-learn.org/stable/auto_examples/index.html)に，代表的なアルゴリズムのサンプルコードがまとまっている．

NumPyとscikit-learnには，日本語で読める書籍やWebページも多くあるので，各自検索されたい．

第12回以降，多くのライブラリが紹介され，混乱することもあるかもしれない．ただし，すべての機能を把握している必要はなく，必要に応じて使い方を調べながら活用してほしい．


### 準備
まずは，本講義のプログラムで必要なライブラリをインポートし，データをダウンロードする．ライブラリの説明とインポート方法は，[第12回の講義](#python-modules)を参照されたい．scikit-learnの機能は，線形回帰を実行する節でインポートする．


In [ ]:
import numpy as np  # NumPyのインポート
import matplotlib.pyplot as plt  # 可視化のためのMatplotlibのインポート
%matplotlib inline

In [ ]:
# Matplotlibの日本語設定．詳しくは第12回を参照
# 未インストールの場合は，次の行の先頭の # を外して実行する
# %pip install matplotlib-fontja
import matplotlib_fontja

```{admonition} グラフの日本語が文字化けするとき
:class: caution

第12回と同じく，`matplotlib-fontja` を使って日本語フォントを設定する．`import matplotlib_fontja` によって，日本語フォントが設定される．

Google Colabなどで `ModuleNotFoundError` が出る場合は，上のセルの `# %pip install matplotlib-fontja` の先頭の `#` を外して再実行されたい．`%pip install` は，このノートブックの実行環境にライブラリをインストールする処理である．第12回と同じ実行環境など，既にインストール済みの場合は不要である．

文字化けする場合は，このセルがエラーなく実行できたかを確認されたい．また，その後に別のフォント設定を行うと，設定が上書きされることがある．詳細は第12回の[タイトルなどが文字化けする場合](#matplotlib-mojibake)を参照されたい．
```


以下のセルでは，`wget` を使って，入力データの `data.npy` と，対応する目標出力の `target.npy` をダウンロードする．`.npy` は，NumPyの配列を保存するファイル形式である．

`-O` は保存先のファイル名を指定するオプションである．同じセルを再実行した場合は，指定したファイルを上書きする．ダウンロードに成功したことを確認してから，次の読み込みに進まれたい．


In [ ]:
!wget -O data.npy https://raw.githubusercontent.com/python-hu/resource/main/zero_to_programming/ml_data/lecture/data.npy
!wget -O target.npy https://raw.githubusercontent.com/python-hu/resource/main/zero_to_programming/ml_data/lecture/target.npy

### データの読み込み
続いて，線形回帰に用いるデータ `X` と `y` を読み込む．今回は，NumPyのデータ読み込み関数 `np.load()` を使って，先ほどダウンロードしたファイルから配列を読み込む．


In [ ]:
X = np.load('data.npy')
y = np.load('target.npy')

読み込まれたデータ `X` と `y` は，それぞれ入力データと，その入力に対応する目標出力である．前述の例と照らし合わせると，`X` の各行が入力 $x$，`y` の同じ行が対応する出力 $y$ にあたり，入出力のペア $(x,y)$ が $N$ 個（今回は $N=30$）含まれている．

以下のセルで，読み込まれた値を確認されたい．


In [ ]:
X

In [ ]:
y

実行環境によっては一部省略されるが，`X` と `y` のどちらにも数値が並んでいることが確認できる．これが今回の線形回帰で扱うデータである．

これらのデータは，それぞれ30行1列の二次元配列として保存されている．この配列の形状は `名前.shape` で確認できる．以下のセルを実行されたい．


In [ ]:
print(f'X.shape: {X.shape}, y.shape: {y.shape}')

`X.shape` と `y.shape` は，どちらも `(30, 1)` と表示される．二次元配列では，`shape` は `(行数, 列数)` を表す．この例では，30個の観測データについて，入力と目標出力をそれぞれ1列に並べている．

```{admonition} 入力と目標出力の対応を保つ
:class: tip

`X` の1行目と `y` の1行目，`X` の2行目と `y` の2行目，というように，同じ行同士が一つのペアになっている．片方だけを並べ替えると対応が崩れるため，データを扱うときには注意されたい．
```

また，読み込まれたデータは，NumPyの配列を表す `ndarray` という型で管理される．データ型は `type(名前)` で確認できる．


In [ ]:
print(f'type(X): {type(X)}, type(y): {type(y)}')

ここまでで，利用するデータの読み込みと，形状・型の確認が完了した．`np.load()` はファイルから配列を読み込む関数であり，後述する `np.array()` はリストなどから配列を作る関数である．どちらの場合も，作られる配列の型は `ndarray` となる．NumPyについてもう少し詳しく知りたい受講者は，後述する「【発展】NumPyの使い方」を参考にされたい．


### データの可視化
第12回でも述べたように，データをグラフなどで視覚的に表示してみることは重要である．前述のように数値を並べるだけでは，`X` と `y` の関連性を捉えにくい．ここでもMatplotlibを用いて，`X` と `y` を散布図として可視化する．Matplotlibの使い方は，[第12回の資料](#matplotlib-visualization)を確認されたい．

以下では，横軸を入力 `X`，縦軸を目標出力 `y` とし，一つのペアを一つの点として表示する．`marker='x'` は，点を「×」の形で表示する指定である．


In [ ]:
plt.scatter(X, y, marker='x', label='観測されたデータ') # 散布図のプロット
plt.title('xとyの観測値と関連性') # タイトルを追加
plt.xlabel('x') # x軸のラベルを追加
plt.ylabel('y') # y軸のラベルを追加
plt.legend()

散布図を見ると，`X` が大きくなるにつれて `y` も大きくなる，正の相関が見られる．点はおおむね直線に沿って並んでおり，最初の例のように，$y=ax+b$ で関係を近似できそうである．次の節では，このデータに当てはまる直線の傾きと切片を求める．


### scikit-learnによる線形回帰の学習
では，実際に **scikit-learn** という機械学習のライブラリを用いて，`X` と `y` のペアからなるデータから，$y=ax+b$ の傾き $a$ と切片 $b$ を求めてみよう．

scikit-learnで線形回帰を実行するためには，`LinearRegression` クラスをインポートする必要がある．以下のセルで実行する．


In [ ]:
from sklearn.linear_model import LinearRegression

インポートした `LinearRegression` クラスをインスタンス化し，`reg` に代入する．続いて，入力データ `X` と目標出力 `y` をメソッド `fit` に渡すと，データに当てはまる傾きと切片が計算される．この処理が，ここでの **学習** にあたる．クラスについては，[第10回の講義](../10/class.ipynb)を復習されたい．


In [ ]:
reg = LinearRegression()
reg = reg.fit(X, y)

`reg.fit(X, y)` によって，データから求めた傾きと切片が，インスタンス `reg` に保存される．`fit` は学習したインスタンス自身を返すため，上の例では戻り値を再び `reg` に代入している．`reg.fit(X, y)` と書くだけでも学習結果は保存される．

傾き $a$ と切片 $b$ は，それぞれ `coef_` と `intercept_` という属性として参照できる．第10回でインスタンス変数を参照したときと同じように，`.` を使って値を確認する．末尾の `_` も名前の一部なので，省略しないようにされたい．


In [ ]:
print(f'傾き a = {reg.coef_}, 切片 b = {reg.intercept_}')

傾きと切片を見ると，それぞれ `2` と `1` に近い値が得られている．今回のデータでは，傾きは約 `2.009`，切片は約 `0.973` となる．このように，数値データから，$y=ax+b$ で表される関係性を推定できた．

なお，値が `[[...]]` や `[...]` の形で表示されるのは，配列として保存されているためである．この例では，それぞれに傾きと切片が一つずつ含まれている．


### 結果の確認
最後に，求めた傾きと切片を使って予測値を計算し，観測されたデータと重ねてプロットしてみよう．以下の `a * X + b` は，直線の式 $ax+b$ を各入力に適用する処理である．


In [ ]:
a = reg.coef_
b = reg.intercept_
y_pred = a * X + b

plt.scatter(X, y, marker='x', label='観測されたデータ')  # 散布図のプロット
plt.plot(X, y_pred, c='g', label='線形回帰によって得られた関数')  # 直線のプロット
plt.title('LinearRegressionの結果')  # タイトルの追加
plt.xlabel('x')  # x軸のラベルの追加
plt.ylabel('y')  # y軸のラベルの追加
plt.legend()

```{admonition} predictで予測値を求める
:class: tip

学習済みの `reg` には，予測値を計算する `predict` メソッドもある．`reg.predict(X)` とすると，上の `a * X + b` と同じ予測値が得られる．例えば，入力が `0.5` のときの予測値は `reg.predict([[0.5]])` で求められる．`[[0.5]]` は，1行1列の入力データを表す．
```

```{admonition} データへの当てはまりと予測性能
:class: caution

ここでは，学習に使ったデータと直線を重ねて，当てはまりを確認している．新しいデータに対してどれくらい正確に予測できるかは，この図だけでは判断できない．予測性能を確かめるには，学習に使っていないデータで評価することが大切である．
```

このように，ライブラリを使うと，機械学習・データ分析のアルゴリズムを少ないコードで利用できる．ここまでは用意されたデータを使ったが，自分でデータを作ったり，計算の中身を確かめたりすると，理解をさらに深められる．そこで，前述の線形回帰のプログラムを，

1. NumPyでベクトルや行列を定義し，計算する → 【発展】NumPyの使い方
2. `np.load()` で読み込む代わりに，NumPyの関数を使ってデータを生成する → 【発展】データの生成
3. 自分で生成したデータを使って，scikit-learnによる線形回帰を実行する → 【発展】scikit-learnを利用して生成したデータから線形回帰を解く
4. scikit-learnの代わりに，自身で線形回帰の計算を実行する → 【発展】scikit-learnを利用せずに線形回帰を解く

の4つの手順に分けて説明する．線形代数を学んでいない学生も受講しているため，発展的な内容として位置付けているが，ぜひコードを実行し，結果を確かめながら読み進めてほしい．


```{admonition} scikit-learnを用いた線形回帰のまとめ

- scikit-learnは，機械学習のアルゴリズムがまとめられたライブラリである．
- `LinearRegression` クラスのインスタンスを作成し，`fit(X, y)` で学習する．
- 学習した傾きと切片は，`coef_` と `intercept_` で確認できる．
- 学習後は，求めた直線の式や `predict()` を使って予測値を計算できる．
- ライブラリを利用するときは，入力データの形状やメソッドの使い方を調べることが大切である．
```


---

## 【発展】 NumPyの使い方
前述したように，NumPyは複数の値を持つベクトルや行列を計算するためのライブラリである．ここでは後述する内容を理解するためにNumPyの必要最低限の使い方を紹介する．

### NumPyによるベクトルの定義
線形回帰や機械学習に限らず，データを扱う場面では，複数の値を持つベクトルや行列を扱うことが多々ある．今回の線形回帰の例でも複数の $(x,y)$ のデータからパラメータ $a,b$ を計算する必要があり，このとき，複数の $(x,y)$ を保存したデータをベクトルや行列として扱うと便利である．また，前回の講義ではpandasの`DataFrame`を用いて行と列から構成されるデータ構造を扱った．`DataFrame`は条件に基づく値の操作や統計量の算出を直感的に行えるが，例えば，行列計算をしようと思うと一手間必要となる．

このような場面で便利なのがNumPyの`ndarray`というデータ型である．先ほどは`type`関数で型を確認したが，以降のセルでは`ndarray`を使うとベクトルや行列の計算が簡単に行えることを確認されたい．`np.array`は，この配列を作成する関数である．

では，まずは，要素を一列に並べた一次元配列としてベクトル $\mathbf{a}=[1,2,3]$ を定義しよう（本講義資料では，ベクトルは小文字で太字で，行列は大文字で太字で記述する）．ベクトルや行列は以下で定義できる．

```python
名前 = np.array(要素)
```

まずは NumPyを`import`する．慣習上，`np`と短い名前で`import`する（復習→[モジュールのインポート](#python-modules)）．

In [ ]:
import numpy as np

ベクトルを定義する．

In [ ]:
a = np.array([1, 2, 3])

In [ ]:
a

上記のように，NumPyでは要素としてリスト（またはタプル）を与えて定義する．`type`関数でデータの型を確認されたい（復習→[変数の型と名前](#variable-types-and-names)）．

In [ ]:
type(a)

続いて，配列の形状を`名前.shape`で確認する．

In [ ]:
a.shape

`(3,)`という形状から，3つの要素が一次元配列`a`に含まれていることがわかる．

NumPyでは，一次元配列によるベクトルの表現と，二次元配列による縦ベクトル（列ベクトル）・横ベクトル（行ベクトル）の表現を区別する．線形代数で学んだ縦・横の表記に対応させるために，NumPyで横ベクトル・縦ベクトルを作成する．

以下のセルのようにリストをもう一組の `[]` で囲むと，横ベクトル版の $\mathbf{a}=[1,2,3]$ を1行3列の配列として定義できる．

In [ ]:
b = np.array([[1, 2, 3]])
b

続いて，縦ベクトル $\mathbf{a}=[1,2,3]^\top$ を作成する．

In [ ]:
c = np.array([[1], [2], [3]])
c

慣れるまではわかりにくいだろうが，`shape`でベクトルの形状を確認しながらプログラミングすると良い．

In [ ]:
print(f'a.shape: {a.shape}, b.shape: {b.shape}, c.shape: {c.shape}')

上記のセルのように形状を確認すると，定義した配列の違いがわかる．`a`の形状は`(3,)`であり，3つの値を一次元に並べている．一方，`b`は`(1, 3)`の1行3列，`c`は`(3, 1)`の3行1列の二次元配列である．

```{admonition} 要素数が同じでも形状は異なる
:class: tip

`a`，`b`，`c`の要素数はすべて3だが，形状は異なる．`shape`は各軸の長さを表すタプルであり，要素の総数は`size`で確認できる．行列計算やライブラリへの入力では，値だけでなく形状にも注意されたい．
```

### NumPyによる行列の定義
続いて，同様に`np.array`を利用して行列を作成する．作成すると言っても`np.array`の要素の与え方が異なるだけである．以下の行列

$$\mathbf{A} = \begin{bmatrix}
1 & 2 \\
3 & 4 \\
\end{bmatrix}$$

は次のセルで定義できる．

In [ ]:
A = np.array([[1, 2], [3, 4]])

In [ ]:
A

値が2行2列の配列に格納されていることがわかる．

In [ ]:
A.shape

同様に，`A.shape`とすると行列の行数と列数が取得できる．

### NumPyによる行列計算
ベクトルと行列の定義の次は線形代数で学んだ行列計算をNumPyを用いて実装することを試みる．まずはじめに，次の行列 $\mathbf{A}, \mathbf{B}$ を定義する．

$$
\mathbf{A} = \begin{bmatrix}
1 & 2 \\
3 & 4 \\
\end{bmatrix},
\mathbf{B} = \begin{bmatrix}
5 & 6 \\
7 & 8 \\
\end{bmatrix} \\
$$

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

定義された行列の`shape`や中身の要素は各自確認されたい．以下のセルで行列の和 $\mathbf{A} + \mathbf{B}$ を計算する．

In [ ]:
A + B

引き算も `-` の演算子で計算できる．

In [ ]:
A - B

行列の積  $\mathbf{A}\mathbf{B}$ を計算しよう．行列 $\mathbf{A}, \mathbf{B}$ はそれぞれ2行2列であり，$\mathbf{A}$ の列数と $\mathbf{B}$ の行数が等しいため，行列の積  $\mathbf{A}\mathbf{B}$ が計算できる．行列の積は `np.dot`関数や`np.matmul`関数で計算できる．

In [ ]:
np.dot(A, B)

In [ ]:
np.matmul(A, B)

プログラミングでも線形代数で学んだ計算のルールは同じである．そのため，行数と列数によってはエラーとなることに注意されたい．繰り返しとなるが`shape`でベクトルや行列の形状を把握しておくことが大切である．以降は逆行列など本講義に関連するNumPyの機能をいくつか紹介するが数多くサポートされている行列計算の一部である．線形代数で学んだ計算のほとんどはNumPyで実装可能なので復習も兼ねて各自で調べてみることをおすすめする．

```{admonition} 行列の積と要素ごとの積
:class: tip

二次元配列の`A * B`は，対応する位置の要素同士を掛ける計算である．行列の積を求めるには，上記の`np.dot(A, B)`や`np.matmul(A, B)`を使う．`A @ B`とも書ける．
```

```{admonition} 数式を扱うライブラリ
:class: tip
NumPyだけでなくPythonには数学関連のライブラリが多くある．特に，[SymPy](https://www.sympy.org/en/index.html)はシンボルとして計算式を定義でき，直感的に数式をプログラミングできる．具体例の計算やグラフの可視化が容易であり，抽象的な大学数学を学ぶ助けとなるはずである．
```

## 【発展】 データの生成
NumPyの必要最低限の基礎を紹介した後は，`np.load`関数で事前に用意したデータを利用する代わりに自身でデータを生成することを試みる．後述するように，データ数（`N`），傾き（`a`），切片（`b`），変動量（`noise_scale`）を制御することができ，線形回帰の特性をデータを変更することから確認できるようになる．

先ほどの例と同様に，ここでは第12回の講義のように複数個（$N$個）の $(x,y)$ のペアデータが利用できること，各データの入出力間には $y=2x+1$ という関係性があると仮定し，NumPyを用いて以下の手順で生成する．

まずはデータの個数を定義する．ここでは $N=30$ とする．

In [ ]:
N = 30

続いて，個数 `N` に応じたデータをNumPyの `linspace` 関数を利用して作成する．`numpy` は `np` としてインポートしたので以下のように呼び出すことができる．`linspace`関数のサンプルコードは後述するので確認されたい．

実際に，両端の $0$ と $1$ を含む範囲で，等間隔に並ぶ $N=30$ 個の値を持つベクトルを作成する．

In [ ]:
X = np.linspace(0, 1, N)  # xの値

値を確認する．

In [ ]:
X

NumPyでは `名前.shape` とすると配列の形状がわかる．今回は $N=30$ 個の値を持つ一次元に並べられたベクトルなので `(30,)` と表示される．

In [ ]:
X.shape

最後に `X` に対応する `y` を作成する．ただし，一般的には気温など観測値にノイズが含まれることがある（ここでは `y` が観測値）．ここでは，NumPyの `random` モジュールの `randn` 関数を用いて乱数を生成し，ノイズを表現する．まずはノイズが含まれていないデータ `y_gt` を生成する．

In [ ]:
y_gt = 2 * X + 1  # y=2x+1の真の値

続いて，ノイズが付加された出力 `y` を作成する．`noise_scale`変数はノイズの大きさ（標準偏差）を指定する．各データに加わるノイズの値は，乱数によって異なる．

In [ ]:
noise_scale = 0.1
y = 2 * X + 1 + noise_scale * np.random.randn(N)  # 乱数によるノイズを加えた観測データ

中身を確認する．

In [ ]:
y_gt

In [ ]:
y

ノイズが付加されたため異なる値であることがわかる．ここまでで線形回帰で利用するデータの生成は完了であるが，上記の処理をこの後複数回呼び出す必要があるため関数化しておく．

In [ ]:
def make_data(N, a, b, noise_scale):
    X = np.linspace(0, 1, N)  # xの値
    y_gt = a * X + b  # y=ax+bの真の値
    y = a * X + b + noise_scale * np.random.randn(N)  # 乱数によるノイズを加えた観測データ
    return (X, y, y_gt)

### 補足：`linspace`関数の説明
`linspace` 関数は引数としてスタートの値，最後の値，個数を与える．既定では，スタートと最後の両方の値を含むように等間隔に並べる．以下の例のように，$0$ から $10$ の範囲で $4$ 個の値をもつベクトルを作成できる．

In [ ]:
x = np.linspace(0, 10, 4)
print(x)

### 補足：乱数の生成
`np.random.randn` は 平均0，標準偏差1の**標準正規分布** から引数として与えられる個数だけ乱数を生成する．

In [ ]:
random_value = np.random.randn(3)
print(random_value)

そのため，`noise_scale=0.1`のとき，`noise_scale * np.random.randn(N)` は，生成した $N$ 個の乱数を $0.1$ 倍する処理となる．このノイズは平均0，標準偏差0.1の正規分布に従うが，実際に得られた有限個の値の平均や標準偏差がぴったりその値になるとは限らない．

```{admonition} 実行するたびに結果が変わる
:class: tip

乱数を生成するセルや`make_data`を再度実行すると，異なるノイズが加わるため，データや学習結果も変わる．結果を比べるときは，同じデータを使っているかにも注意されたい．
```

## 【発展】 scikit-learnを利用して生成したデータから線形回帰を解く
先ほどの例と同様にscikit-learnを利用して線形回帰を解く．ただし，`np.load`関数で得られたデータ `X`と`y`ではなく自身で生成したデータ `X`と`y`に対して行う．`LinearRegression`クラスの使い方は先ほどと同様であるが，入力前にデータを加工する必要があるので注意されたい（これは実装された`LinearRegression`クラスの仕様によるものである）．

再度，線形回帰の前提について復習しておくと，今から線形回帰 `LinearRegression` を用いて，$(x,y)$ から $y=ax+b$ の $a$ と $b$ を計算することを試みる．前述のセルで作成したデータで考えると，$x$ は `X`，$y$ は `y` に対応する．今回ノイズを加えているのは`y`のみである．データを生成した我々は`y_gt`や`a=2`，`b=1`を知っているが，学習に渡すのは`X`と`y`のみである．これらのデータから元の傾きと切片に近い値を求められるかを確認する．

### データの生成と加工
まずは【発展】データ生成で作成した `make_data` 関数を用いてデータを生成する．

In [ ]:
X, y, y_gt = make_data(30, 2, 1, 0.1)

続いて，作成されたデータ`X`，`y`は一次元配列であるため，先ほどの例と同じ形状の縦ベクトルへ変換する．`LinearRegression`の入力`X`は，`(データ数, 特徴量の数)`という二次元配列にする必要がある．今回は特徴量が $x$ の1つなので，形状を`(30,)`から`(30, 1)`へ変える．

一方，出力が1つの場合，`y`は一次元配列のままでも学習できる．ここでは先ほどの例に合わせて，`y`も`(30, 1)`にする．

In [ ]:
X = X[:, np.newaxis]
y = y[:, np.newaxis]

`np.newaxis`の説明は後述する．

### 線形回帰モデルの当てはめ
作成したデータで線形回帰を行うために，再度，先ほどの`LinearRegression` クラスをインスタンス化し，(`X,y`)のペアをインスタンスメソッド `fit` に渡す処理を再度行う．

In [ ]:
reg = LinearRegression()
reg = reg.fit(X, y)

### 結果の確認と可視化

In [ ]:
print(f'傾き a = {reg.coef_}, 切片 b = {reg.intercept_}')

傾きと切片をみると，データを生成するときに設定した `2` と `1` に近い値が得られることが期待される．この結果はデータの関係性，データ数 $N$，ノイズの大きさなどの要因で変化するので一度自分で確かめてみると良い．

値を確認するだけでなく実際に `X` に対して計算することでグラフのプロットを試みる．計算された値がノイズがない真の関係性 $y=2x+1$ にどの程度近いかを確認されたい．

In [ ]:
a = reg.coef_
b = reg.intercept_
y_pred = a * X + b

plt.scatter(X, y, marker='x', label='観測されたデータ')
plt.plot(X, y_gt, c='r', linestyle='--', label='ノイズがない真の関係性')
plt.plot(X, y_pred, c='g', label='線形回帰によって得られた関数')
plt.title('LinearRegressionの結果')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

```{admonition} ライブラリのソースコードを読む
:class: tip
`LinearRegression`クラスの中身はscikit-learnの[リファレンスページ](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)へアクセスし，ページ中にある `[source]`リンク からGitHub上で公開されているコードを見ることができる．中身を見るとこれまでの講義で学んだ[条件分岐](../04/conditional_branch.ipynb)，[関数](../08/functions_and_scope.ipynb)，[クラス](../10/class.ipynb)といった機能で書かれていることがわかる．ゼロから学ぶプログラミングが，このようなプログラムを読み解けるヒントになるだけでなく，自身でソフトウェアを開発して公開するきっかけになれば幸いである．
```

### 補足：`np.newaxis`
`np.newaxis`は，インデキシングの中で軸を追加するために使う．ここでの`X[:, np.newaxis]`は，`:`で元の要素をすべて取り出し，その後ろに長さ1の軸を追加する．これにより，形状`(30,)`の一次元配列が`(30, 1)`の縦ベクトルになる．実際に以下の例で確認されたい．

In [ ]:
a = np.linspace(0, 5, 5)
print(f'ベクトル a の中身は {a} であり，形状は {a.shape} である．')

b = a[:, np.newaxis]
print(f'newaxisしたベクトル b の中身は {b} であり，形状は {b.shape} である．')

NumPyによるベクトルの定義で用いた例を使うと以下のセルとなる．縦ベクトルになるように再度ベクトルを定義することなく変換できていることがわかる．

In [ ]:
a = np.array([1, 2, 3])
print(f'ベクトル a の中身は {a} であり，形状は {a.shape} である．')

b = a[:, np.newaxis]
print(f'newaxisしたベクトル b の中身は {b} であり，形状は {b.shape} である．')

## 【発展】 scikit-learnを利用せずに線形回帰を解く
利用するだけなら，scikit-learnは便利なライブラリである．ここでは，敢えて，scikit-learnを使わず，NumPyを用いて，数式に基づいて線形回帰を解く方法を紹介する．

そのために，$\mathbf{X}^\top\mathbf{X}$ の逆行列が存在する場合，線形回帰の解析解が以下の式で与えられることを利用する．

$$
\mathbf{W} = \left( \mathbf{X}^\top \mathbf{X} \right)^{-1} \mathbf{X}^\top \mathbf{y}
$$

この式は，次の **正規方程式（Normal Equation）** を解くことで得られる．

$$
\mathbf{X}^\top\mathbf{X}\mathbf{W}=\mathbf{X}^\top\mathbf{y}
$$

この正規方程式は，予測値と目標の出力値（目的変数）の差の二乗和を最小にする，最小二乗法から導出できる．そのアイデアは，二乗誤差をパラメータの関数として表し，最小となるパラメータを偏微分から求めるというものである．より簡単にいうと，高校で学んだ関数の最小値を求めるときと同じように，関数をパラメータで微分し，0となるようなパラメータを求めている．以下に正規方程式の導出を記載したので興味があれば読んでみて欲しい．

```{admonition} 正規方程式の導出
:class: tip, dropdown

$D$次元のベクトル $\mathbf{x}$ とその目的変数 $y$ からなるペアを $N$ 個与えられ，そのサンプル集合 $\{\mathbf{x}_{i},y_{i}\}^{N}_{i=1}$ を最もよく表す線形モデルのパラメータ $\mathbf{W}$ は$\mathbf{X}^\top\mathbf{X}$ の逆行列が存在する場合，次の式で求めることができる．

$$
\mathbf{W} = \left( \mathbf{X}^\top \mathbf{X} \right)^{-1} \mathbf{X}^\top \mathbf{y}
$$

この式を導出するためにはじめにサンプル集合を行列として表記すると，

$$
\mathbf{X}=\left[\begin{array}{ccccc}
1 & x_{11} & x_{12} & \ldots & x_{1 D} \\
1 & x_{21} & x_{22} & \ldots & x_{2 D} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_{N 1} & x_{N 2} & \ldots & x_{N D}
\end{array}\right]
$$

となり，各行（第 $i$ 行）は $i$ 番目の観測データ $\mathbf{x}_i$ を示している．一列目の $1$ は切片（バイアス項とも呼ばれる）を式に含めるために追加しており，この行列は $N \times (D+1)$ のサイズを持つことに注意されたい．

目的変数は $N$ 個の値を持つベクトルとして次のように書くことができる．

$$
\mathbf{y}=\left[\begin{array}{c}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{array}\right]
$$

ここで $y_i$ は $i$ 番目の観測データの目的変数である．

関数 $f_{\theta}$ のパラメータ $\mathbf{W}$ は切片を含むことに注意すると $D+1$ 次元のベクトルとして次のように定義できる．

$$
\mathbf{W}=\left[\begin{array}{c}
w_0 \\
w_1 \\
\vdots \\
w_D
\end{array}\right]
$$

ここで $w_0$ は切片に対応するパラメータであり，$w_1$ から $w_D$ は観測データの各次元に対応する．

行列で表記した $\mathbf{X}, \mathbf{y}, \mathbf{W}$ を使って予測値 $\hat{\mathbf{y}}$ を表すと次のように書くことができる．

$$
\hat{\mathbf{y}} = \mathbf{X}\mathbf{W}
$$

実際にこの行列を計算してみると以下のように

$$
\begin{aligned}
\hat{y}_1 & =w_0+w_1 x_{11}+w_2 x_{12}+\cdots+w_D x_{1 D} \\
\hat{y}_2 & =w_0+w_1 x_{21}+w_2 x_{22}+\cdots+w_D x_{2 D} \\
& \vdots \\
\hat{y}_N & =w_0+w_1 x_{N 1}+w_2 x_{N 2}+\cdots+w_D x_{N D}
\end{aligned}
$$

最初の説明で登場した $y=ax+b$ のように各予測値が入力データとパラメータとの線形結合になっていることが確認できる．

線形回帰では目的変数 $\mathbf{y}$ と回帰された予測値 $\mathbf{X}\mathbf{W}$ の差の二乗和が最小になるようにパラメータ $\mathbf{W}$ を求める．この差の二乗和は二乗誤差関数として呼ばれ，次のように定義される．

$$
J(\mathbf{W})=(\mathbf{y}-\mathbf{X} \mathbf{W})^{\top}(\mathbf{y}-\mathbf{X} \mathbf{W})
$$

要素ごとに展開してみると，$i$番目の誤差 $e_i$ の計算は次のようになっており，

$$
e_i=y_i-\left(w_0+w_1 x_{i 1}+w_2 x_{i 2}+\cdots+w_D x_{i D}\right)
$$

二乗誤差は，この誤差を二乗して全てのデータにわたって合計したものとなる．

$$
J(\mathbf{W})=\sum_{i=1}^N e_i^2=\sum_{i=1}^N\left(y_i-\left(w_0+w_1 x_{i 1}+w_2 x_{i 2}+\cdots+w_D x_{i D}\right)\right)^2
$$

そして，この二乗誤差関数が最小となるパラメータ $\mathbf{W}$ を求めたいので，パラメータ $\mathbf{W}$ で偏微分し，その導関数が0となるように方程式を解けば良い．

まず二乗誤差について整理すると

$$
\begin{align}
(\mathbf{y}-\mathbf{X} \mathbf{W})^\top(\mathbf{y}-\mathbf{X} \mathbf{W}) &= \left(\mathbf{y}^\top-\mathbf{W}^\top \mathbf{X}^\top\right)(\mathbf{y}-\mathbf{X} \mathbf{W}) \\
&=\mathbf{y}^\top \mathbf{y}-\mathbf{W}^\top \mathbf{X}^\top \mathbf{y}-\mathbf{y}^\top \mathbf{X} \mathbf{W}+\mathbf{W}^\top \mathbf{X}^\top \mathbf{X} \mathbf{W}
\end{align}
$$

となり，これをパラメータ $\mathbf{W}$ で偏微分したときに 0 となる方程式を立てる．

$$
\frac{\partial(\mathbf{y}-\mathbf{X} \mathbf{W})^\top(\mathbf{y}-\mathbf{X} \mathbf{W})}{\partial \mathbf{W}}=-\frac{\partial \mathbf{W}^\top \mathbf{X}^\top \mathbf{y}}{\partial \mathbf{W}}-\frac{\partial \mathbf{y}^\top \mathbf{X} \mathbf{W}}{\partial \mathbf{W}}+\frac{\partial \mathbf{W}^\top \mathbf{X}^\top \mathbf{X} \mathbf{W}}{\partial \mathbf{W}}=0
$$

ベクトル・行列の微分に注意して計算すると次の等式が得られ，

$$
\mathbf{X}^\top \mathbf{y} = \mathbf{X}^\top \mathbf{X} \mathbf{W}
$$

この等式が正規方程式である．$\mathbf{X}^\top\mathbf{X}$ の逆行列が存在するとき，左からその逆行列を掛けると次の解が得られる．

$$
\mathbf{W} = \left( \mathbf{X}^\top \mathbf{X} \right)^{-1} \mathbf{X}^\top \mathbf{y}
$$

以上，線形代数とベクトル・行列の微分の知識より正規方程式の導出をすることができた．

```

式の導出をすべて理解できなくても，上記の式とデータ $\mathbf{X}, \mathbf{y}$ があれば，傾きと切片のパラメータ $\mathbf{W}$ をNumPyで計算できる．線形代数で学んだ転置，行列の積，逆行列がコードとどのように対応するかを確認してみよう．

### データの生成と加工
先ほどの【発展】scikit-learnを利用して生成したデータから線形回帰を解くと同様に，以下のようにデータを生成・加工する．

In [ ]:
X_, y, y_gt = make_data(30, 2, 1, 0.1)

In [ ]:
X_ = X_[:, np.newaxis]
y = y[:, np.newaxis]

ここで，切片 $b$ を含めて計算するために，すべての要素が1である列を入力データに追加する．`LinearRegression`では切片の処理が用意されているが，今回は数式に合わせて自分で準備する．

In [ ]:
X = np.hstack((X_, np.ones((X_.shape[0], 1))))

上記の処理の意味は，式で書くと，$y=ax+b$ を $y=ax_0+bx_1$ （ただし，$x_0=x, x_1=1$）という形で書き換えている．これは傾きと切片を同じパラメータベクトル $\mathbf{W}$ で表現するためである．そのため，インデキシングを用いて修正前の行列 `X_` と修正後の行列 `X` の5番目のデータ (index=4) を参照すると以下のようになっている．

```{admonition} 列の順番とパラメータの順番
:class: tip

上の導出では1の列を先頭に置いたが，このコードでは末尾に追加している．そのため，`W`の並びは傾き，切片の順になる．列の順番を変える場合は，対応するパラメータの順番も合わせる必要がある．`X_.shape[0]`はデータの行数であり，同じ行数の1の列を作成するために使っている．
```

In [ ]:
print(X_[4])
print(X[4])

$x_0$, $x_1$が$x_0=x, x_1=1$となっていることがわかる．ここまででデータの準備は完了である．

#### 補足：`np.hstack`
`np.hstack`関数の処理を以下のコードで確認されたい．まず，元の $x_0$ に相当する配列を作成する．

In [ ]:
M = 5
a = np.linspace(0, 5, M)
b = a[:, np.newaxis]

In [ ]:
b

続いて，すべて `1` が格納された `M` 個の要素を持つベクトルを作成する．これは`np.ones`で作成できる．

In [ ]:
c = np.ones((M, 1))

作成した `b` と `c` を `np.hstack` で結合する．ただし，

```
b = array([[0.  ],
          [1.25],
          [2.5 ],
          [3.75],
          [5.  ]])

c = array([[1.],
           [1.],
           [1.],
           [1.],
           [1.]])
```
である．

In [ ]:
d = np.hstack((b, c))

上記のサンプルコードの`d`は，`b`と`c`を横方向に結合した5行2列の配列である．先ほどの`X`も同じ方法で，$y=ax_0+bx_1$（ただし，$x_0=x, x_1=1$）に対応する2列の配列にしている．二次元配列を横に結合するときは，行数をそろえる必要がある．

### 正規方程式から得られた解析解の計算
線形回帰の解析解は以下で導出できた．

$$
\mathbf{W} = \left( \mathbf{X}^\top \mathbf{X} \right)^{-1} \mathbf{X}^\top \mathbf{y}
$$

プログラムコードの可読性の観点から，上記の式を以下のように分割して計算する．

$$
\mathbf{W}_1 = \left( \mathbf{X}^\top \mathbf{X} \right)^{-1} \\
\mathbf{W}_2 = \mathbf{X}^\top \mathbf{y} \\
\mathbf{W} = \mathbf{W}_1 \mathbf{W}_2
$$

まず，$\mathbf{W}_1$ を計算する．以下のコードで実装できる．

In [ ]:
W_1 = np.linalg.inv(np.dot(X.T, X))

続いて，$\mathbf{W}_2$ を計算する．以下のコードで実装できる．

In [ ]:
W_2 = np.dot(X.T, y)

最後に，$\mathbf{W}$ を計算する．以下のコードで実装できる．

In [ ]:
W = np.dot(W_1, W_2)

上記の処理は1行で書くこともできる．

In [ ]:
W = np.dot(np.linalg.inv(np.dot(X.T, X)), np.dot(X.T, y))

#### 補足：行列の転置
NumPyでは，行列の転置は `行列.T` でできる．以下のサンプルコードで確認されたい．

In [ ]:
A = np.array([[0, 1], [2, 3]])
A

In [ ]:
A.T

#### 補足：逆行列の計算
逆行列の計算は `np.linalg.inv` 関数を利用すれば良い．

In [ ]:
A = np.array([[0, 1], [2, 3]])
A_inv = np.linalg.inv(A)
A_inv

```{admonition} 逆行列を使う計算の注意点
:class: caution

`np.linalg.inv`では，逆行列が存在しない特異行列の場合にエラーとなることがある．また，逆行列が計算できても，行列によっては丸め誤差の影響が大きくなり，結果が不正確になる場合がある．今回逆行列を求めるのは $\mathbf{X}^\top\mathbf{X}$ という2行2列の行列だが，特徴量が増えるとこの行列も大きくなる．

ここでは数式とコードの対応を学ぶために逆行列を使った．実際の最小二乗問題では，逆行列を明示的に計算せずに解を求める`np.linalg.lstsq(X, y, rcond=None)`などの方法も利用できる．
```

### 傾きと切片の確認
計算された `W` について，傾きと切片を確認する．

In [ ]:
W

上記の結果からわかるように，傾きと切片の参照はインデキシングが必要である．以下のように参照する．

In [ ]:
coef = W[0]
coef

In [ ]:
intercept = W[1]
intercept

ここで生成した同じデータ`X_`と`y`を用いて，`LinearRegression`でも学習し直し，結果を比較する．先ほどの`reg`は別の乱数で生成したデータから学習したものなので，そのままでは同じ条件で比較できない．`LinearRegression`には1の列を追加する前の`X_`を渡す．

In [ ]:
reg = LinearRegression()
reg = reg.fit(X_, y)
reg.coef_

In [ ]:
reg.intercept_

同じデータを使うと，`LinearRegression`の結果と，丸め誤差の範囲で一致する値が得られる．計算方法は異なるが，どちらも同じ二乗誤差を最小にする傾きと切片を求めている．

### 結果のプロット
最後に，計算された結果をグラフでプロットする．

In [ ]:
a = W[0]
b = W[1]
y_pred = a * X_ + b

plt.scatter(X_, y, marker='x', label='観測されたデータ')
plt.plot(X_, y_gt, c='r', linestyle='--', label='ノイズがない真の関係性')
plt.plot(X_, y_pred, c='g', label='線形回帰によって得られた関数')
plt.title('正規方程式の結果')
plt.xlabel('x')  # x軸のラベルを追加
plt.ylabel('y')  # y軸のラベルを追加
plt.legend()

scikit-learnを用いずに線形回帰を実行することができた．Pythonには機械学習だけでなく，高度なアルゴリズムを提供するライブラリが多くあり，便利なプログラミング言語である．これらは内部のアルゴリズムをすべて理解していなくても利用できるが，仕組みを学ぶことで，使い方や結果への理解を深められる．Pythonの第一歩としてのゼロからはじめるプログラミングでは，これらの機能を利用するための基礎を学んだ．受講者の皆さんにはぜひ内部のアルゴリズムまで調べて，今回のように**数式をもとに** 自身で実装してみることを推奨する．このような経験は線形代数・微積分・確率統計など大学数学を学ぶモチベーションとなるだけでなく，皆さんがこれから学ぶプログラミング，数学，加えて専門科目の理解をより深めてくれるはずである．